# Criteo Uplift: who actually changes their mind?

**The question.** An advertiser can only reach a limited number of people. The obvious strategy is to target whoever is most likely to convert. This notebook argues — and then checks against roughly 14 million rows of real randomized experiment data — that this is usually the wrong strategy, and that the useful question is *whose behaviour changes because they were shown the ad*.

**The data.** CRITEO-UPLIFTv2.1 is a large randomized advertising experiment released publicly by Criteo. Each row has twelve anonymous numeric features, a randomly assigned treatment flag, and two binary outcomes.

**What you will learn**

1. Why ranking by conversion probability and ranking by treatment effect are different problems
2. What one row of this dataset is — and what it is not
3. How lopsided the treatment/control split is, and why that matters
4. How rare conversion is, and what that does to precision
5. What the twelve anonymous features look like
6. Whether the treated and control groups are actually comparable
7. How many rows are indistinguishable, and whether numeric precision creates that
8. What all of this implies for the uplift models that come next

**What is not here.** No model is trained and nothing is predicted. This is the descriptive groundwork that comes *before* modeling, and it is deliberately kept separate from it.

## 1. Why response modeling and uplift modeling are different questions

Response modeling estimates **P(conversion | features)** — how likely is this person to convert?

Uplift modeling estimates **E[conversion if treated] − E[conversion if not treated]** among people with the same features — how much does treating this person *change* the outcome?

These two quantities rank people differently, and the difference is not subtle. Consider two customers:

- **Customer A** converts with probability 0.90 if shown the ad, and 0.89 if not. She was going to buy anyway. Her uplift is about +1 percentage point.
- **Customer B** converts with probability 0.20 if shown the ad, and 0.10 if not. His uplift is +10 percentage points.

A response model ranks A far above B. An uplift model ranks B far above A. Spend the budget on A and most of it buys conversions that would have happened regardless.

The only reason we can ask this question at all is that treatment here was **randomly assigned**. Without randomization, treated and untreated people would differ systematically, and the gap between their conversion rates would reflect who they are rather than what the ad did.

In [ ]:
# Illustrative example only — these four customers are hypothetical and are NOT from the Criteo data.
import pandas as pd
from IPython.display import Markdown, display

illustration = pd.DataFrame({
    'customer': ['A', 'B', 'C', 'D'],
    'p_convert_if_treated': [0.90, 0.20, 0.55, 0.05],
    'p_convert_if_not_treated': [0.89, 0.10, 0.50, 0.01],
})
illustration['uplift'] = (
    illustration['p_convert_if_treated'] - illustration['p_convert_if_not_treated']
)
illustration['rank_by_response'] = illustration['p_convert_if_treated'].rank(ascending=False).astype(int)
illustration['rank_by_uplift'] = illustration['uplift'].rank(ascending=False).astype(int)

display(illustration)

display(Markdown('''
**What this shows.** The two rankings disagree almost completely. Customer A is the best target by
conversion probability and the *worst* target by uplift; customer B is the reverse.

**What this does not show.** Nothing yet about the Criteo data — these are invented numbers, used
only to make the distinction concrete.

**Why it matters next.** Everything below asks whether this dataset can actually support the second
kind of ranking: is the assignment usable as a randomized comparison, and is there enough signal to
estimate an effect rather than just a propensity?
'''))

In [ ]:
# Setup: locate the dataset and open it with DuckDB.
# Runs on Kaggle (with the Criteo uplift dataset attached) and locally, with no other dependencies.
import math
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

SEARCH_ROOTS = [Path('/kaggle/input'), Path('data/raw'), Path('../data/raw'), Path('.')]

def find_criteo_file():
    for root in SEARCH_ROOTS:
        if not root.is_dir():
            continue
        for pattern in ('**/*.csv.gz', '**/*.csv'):
            for candidate in sorted(root.glob(pattern)):
                if 'criteo' in candidate.name.lower():
                    return candidate
    raise FileNotFoundError(
        'Could not find the Criteo uplift CSV. On Kaggle, attach the Criteo uplift dataset; '
        'locally, place it under data/raw/.'
    )

DATA_PATH = find_criteo_file()
sql_path = DATA_PATH.as_posix().replace(chr(39), chr(39) * 2)  # escape any quote in the path

FEATURES = [f'f{i}' for i in range(12)]
TREATMENT = 'treatment'
OUTCOME = 'conversion'
SECONDARY = 'visit'
AUDIT_ONLY = 'exposure'

con = duckdb.connect(database=':memory:')
con.execute('SET threads = 4')
con.execute('SET memory_limit = ' + chr(39) + '6GB' + chr(39))
con.execute(f'CREATE VIEW data AS SELECT * FROM read_csv_auto({chr(39)}{sql_path}{chr(39)})')

print(f'Reading: {DATA_PATH}')
print(f'DuckDB version: {duckdb.__version__}')

## 2. The dataset: what a row is, and what it is not

Each row is **one released observation from the experiment**. That phrasing is deliberately narrow:

- A row is **not** a person. The public schema has no user identifier, so two identical rows might be two different people, or the same person twice — the data cannot tell us, and neither can we.
- The twelve features `f0`–`f11` are **anonymous**. Criteo does not publish what they represent. Any story about what `f3` means would be invented, so none is offered here.

The roles of the fields are fixed for the whole analysis:

| Field | Role |
|---|---|
| `f0`–`f11` | the only model inputs |
| `treatment` | randomly assigned; 1 = ad shown, 0 = not shown |
| `conversion` | the primary outcome |
| `visit` | a secondary outcome, analysed separately; never an input |
| `exposure` | whether the ad was actually seen — diagnostic only, never an input |

**Why `exposure` is never an input, and never a filter.** Whether someone actually saw the ad happens *after* assignment and depends on their own behaviour. Using it as a feature, or restricting the analysis to exposed rows, would quietly swap a randomized comparison for a self-selected one. Throughout this notebook the effect of interest is the effect of *being assigned* the treatment — not the effect of actually seeing the ad.

In [ ]:
# Schema, size, and completeness of the fields we rely on.
schema = con.execute('DESCRIBE data').df()[['column_name', 'column_type']]

all_columns = schema['column_name'].tolist()
present = [c for c in FEATURES + [TREATMENT, OUTCOME, SECONDARY, AUDIT_ONLY] if c in all_columns]

missing_exprs = ', '.join(f'COUNT(*) - COUNT({c}) AS {c}' for c in present)
counts = con.execute(f'SELECT COUNT(*) AS n_rows, {missing_exprs} FROM data').df()
total_n = int(counts.loc[0, 'n_rows'])
missing = counts[present].iloc[0]

roles = {f: 'model input' for f in FEATURES}
roles[TREATMENT] = 'randomized assignment'
roles[OUTCOME] = 'primary outcome'
roles[SECONDARY] = 'secondary outcome (never an input)'
roles[AUDIT_ONLY] = 'diagnostic only (never an input)'

overview = pd.DataFrame({
    'field': present,
    'stored type': [schema.loc[schema['column_name'] == c, 'column_type'].iloc[0] for c in present],
    'role': [roles[c] for c in present],
    'missing values': [int(missing[c]) for c in present],
})
display(overview)

display(Markdown(f'''
**What this shows.** The file holds **{total_n:,} rows** and **{len(all_columns)} columns**. Every
field the analysis depends on is present, and the total number of missing values across all of them
is **{int(missing.sum()):,}**.

**What this does not show.** A complete, well-typed table says nothing about whether the experiment
was run correctly, or whether the features were measured before treatment was assigned. Those are
claims about the study design, not about the file.

**Why it matters next.** Because nothing is missing, every count and rate below is computed on the
full population rather than on whatever survived a cleaning step, so no row is silently dropped.
'''))

## 3. How is treatment allocated?

In a textbook A/B test the two arms are roughly equal in size. This experiment is not built that way. The size of each arm places a hard ceiling on how precisely anything about that arm can be estimated, so it is worth knowing before any modeling starts.

In [ ]:
# Assignment counts and the treated share.
allocation = con.execute(f'''
    SELECT COUNT(*) AS n,
           COUNT_IF({TREATMENT} = 1) AS treated_n,
           COUNT_IF({TREATMENT} = 0) AS control_n,
           AVG({TREATMENT}) AS treated_share
    FROM data
''').fetchone()
n, treated_n, control_n, treated_share = allocation

display(pd.DataFrame({
    'group': ['Control (not shown the ad)', 'Treated (shown the ad)'],
    'rows': [control_n, treated_n],
    'share of all rows': [1 - treated_share, treated_share],
}))

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.bar(['Control', 'Treated'], [control_n, treated_n], color=['#4C78A8', '#F58518'])
ax.set_ylabel('Number of rows')
ax.set_title('How many rows are in each experimental group')
ax.ticklabel_format(axis='y', style='plain')
for i, value in enumerate([control_n, treated_n]):
    ax.text(i, value, f'{value:,}', ha='center', va='bottom')
fig.tight_layout()
plt.show()

display(Markdown(f'''
**What this shows.** The split is heavily lopsided: **{treated_share:.1%}** of rows were treated and
**{1 - treated_share:.1%}** were not. The control group holds **{control_n:,}** rows against
**{treated_n:,}** treated.

**What this does not show.** An uneven split is not a flaw and is not evidence that anything went
wrong — experiments are often designed this way. It also does not tell us whether assignment was
genuinely random; that is examined separately in section 6.

**Why it matters next.** Whichever arm is smaller limits precision. Any method that fits a separate
model per arm inherits that imbalance directly, so the two halves of such a model will not be
equally reliable.
'''))

## 4. How rare is conversion?

Two different things are easy to confuse: how lopsided the *treatment* split is, and how rare the *outcome* is. They have separate consequences and deserve separate measurement.

We also check that all four combinations of treatment and conversion actually occur. If any of the four were empty, entire comparisons would be impossible rather than merely imprecise.

In [ ]:
# The 2x2 table of assignment against outcome, plus within-arm conversion rates.
joint = con.execute(f'''
    SELECT {TREATMENT} AS treatment, {OUTCOME} AS conversion, COUNT(*) AS n
    FROM data GROUP BY 1, 2 ORDER BY 1, 2
''').df()

joint['arm_n'] = joint.groupby('treatment')['n'].transform('sum')
joint['within_arm_rate'] = joint['n'] / joint['arm_n']
display(joint)

control_rate = float(joint[(joint.treatment == 0) & (joint.conversion == 1)]['within_arm_rate'].iloc[0])
treated_rate = float(joint[(joint.treatment == 1) & (joint.conversion == 1)]['within_arm_rate'].iloc[0])
overall_rate = float(joint[joint.conversion == 1]['n'].sum()) / total_n
all_four_cells_present = bool(len(joint) == 4 and (joint['n'] > 0).all())

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.bar(['Control', 'Treated'], [control_rate, treated_rate], color=['#4C78A8', '#F58518'])
ax.set_ylabel('Share of the group that converted')
ax.set_title('Conversion rate within each experimental group')
for i, value in enumerate([control_rate, treated_rate]):
    ax.text(i, value, f'{value:.4%}', ha='center', va='bottom')
ax.set_ylim(0, max(control_rate, treated_rate) * 1.25)
fig.tight_layout()
plt.show()

display(Markdown(f'''
**What this shows.** Conversion is rare: **{overall_rate:.4%}** of all rows. Within groups it is
**{control_rate:.4%}** for control and **{treated_rate:.4%}** for treated, a raw gap of
**{(treated_rate - control_rate) * 100:.4f} percentage points**. All four combinations of treatment
and conversion are populated: **{all_four_cells_present}**.

**What this does not show.** That gap is a descriptive comparison of two averages. It is not a
per-person effect, and it says nothing about *who* the effect is concentrated in — which is the
entire point of uplift modeling. No individual in this table has a measured effect of their own.

**Why it matters next.** Rarity is the binding constraint. With conversion this uncommon, the number
of positive examples — not the number of rows — decides how finely the data can be sliced before the
estimates become noise.
'''))

## 5. What do the twelve anonymous features look like?

We cannot say what these features *mean*, but we can say how they *behave*: their range, where their mass sits, how many distinct values they take, and whether anything is missing.

Cardinality is worth attention. A feature with a handful of distinct values behaves like a category; one with millions behaves like a continuous measurement. Tree-based models split them very differently.

Columns marked with a tilde use DuckDB approximate aggregates, which is what makes a full pass over roughly 14 million rows fast. Minimum, maximum, and missing counts are exact.

In [ ]:
# Per-feature shape: cardinality, range, and where the mass sits.
QUANTILES = [0.01, 0.25, 0.50, 0.75, 0.99]

exprs = []
for f in FEATURES:
    exprs += [
        f'approx_count_distinct({f}) AS {f}__nunique',
        f'MIN({f}) AS {f}__min',
        f'approx_quantile({f}, {QUANTILES}) AS {f}__q',
        f'MAX({f}) AS {f}__max',
        f'COUNT(*) - COUNT({f}) AS {f}__missing',
    ]
cursor = con.execute('SELECT ' + ', '.join(exprs) + ' FROM data')
values = dict(zip([d[0] for d in cursor.description], cursor.fetchone()))

feature_summary = pd.DataFrame([{
    'feature': f,
    'distinct values ~': values[f + '__nunique'],
    'missing': values[f + '__missing'],
    'min': values[f + '__min'],
    'q01 ~': values[f + '__q'][0],
    'q25 ~': values[f + '__q'][1],
    'median ~': values[f + '__q'][2],
    'q75 ~': values[f + '__q'][3],
    'q99 ~': values[f + '__q'][4],
    'max': values[f + '__max'],
} for f in FEATURES])
display(feature_summary)

low_cardinality = feature_summary[feature_summary['distinct values ~'] <= 50]['feature'].tolist()
high_cardinality = feature_summary[feature_summary['distinct values ~'] > 100000]['feature'].tolist()

display(Markdown(f'''
**What this shows.** The twelve features are not interchangeable. Roughly category-like features,
with 50 or fewer distinct values: **{low_cardinality or 'none'}**. Effectively continuous features,
with over 100,000 distinct values: **{high_cardinality or 'none'}**. No feature has missing values.

**What this does not show.** Nothing about meaning. A feature with four distinct values might be a
device type, or a coarse bucket of something continuous — there is no way to tell, and guessing
would be fiction. The tilde columns are approximations accurate to within a small tolerance, not
exact counts.

**Why it matters next.** These shapes decide how each feature should be handled downstream, and the
absence of missing values means no imputation rule has to be chosen — one fewer decision that could
accidentally be applied differently to the treated and control arms.
'''))

## 6. Do the treated and control groups look alike?

Random assignment should make the two arms statistically similar on every feature measured before treatment. Checking this is worthwhile — a large, systematic gap is a strong hint that something upstream is broken — but it is important to be precise about what a *favourable* result means.

Three complementary views, because each misses something the others catch:

- **Standardized mean difference (SMD)** — the gap between arm means expressed in pooled standard deviations, so features on wildly different scales become comparable.
- **Variance ratio** — whether the arms have similar spread, not just similar centres.
- **Kolmogorov–Smirnov (KS) distance** — the largest gap between the two arms cumulative distributions, which detects differences in distributional *shape* that means and variances both miss.

A dashed ±0.10 line appears on the plot below. It is a rule of thumb borrowed from the observational-studies literature, drawn purely for orientation. **Nothing here passes or fails because of it.**

And the caveat that matters most: balance does **not** prove randomization. Arms that look alike are *consistent with* random assignment, which is a considerably weaker statement than proof. With roughly 14 million rows, conventional significance tests would flag differences far too small to matter, so they are not used.

In [ ]:
# Balance: standardized mean difference, variance ratio, median gap, and empirical KS distance.
moment_exprs = []
for f in FEATURES:
    moment_exprs += [
        f'AVG({f}) AS {f}__mean',
        f'VAR_SAMP({f}) AS {f}__var',
        f'approx_quantile({f}, 0.5) AS {f}__median',
    ]
cursor = con.execute(
    f'SELECT {TREATMENT}, ' + ', '.join(moment_exprs) +
    f' FROM data GROUP BY {TREATMENT} ORDER BY {TREATMENT}'
)
names = [d[0] for d in cursor.description]
by_arm = {int(row[0]): dict(zip(names[1:], row[1:])) for row in cursor.fetchall()}

def ks_distance(feature):
    '''Largest gap between the treated and control cumulative distributions.'''
    return con.execute(f'''
        WITH value_counts AS (
            SELECT {feature} AS value,
                   COUNT_IF({TREATMENT} = 0)::DOUBLE AS n_control,
                   COUNT_IF({TREATMENT} = 1)::DOUBLE AS n_treated
            FROM data
            WHERE {feature} IS NOT NULL
            GROUP BY {feature}
        ), cumulative AS (
            SELECT value,
                   SUM(n_control) OVER (ORDER BY value ROWS UNBOUNDED PRECEDING)
                       / SUM(n_control) OVER () AS cdf_control,
                   SUM(n_treated) OVER (ORDER BY value ROWS UNBOUNDED PRECEDING)
                       / SUM(n_treated) OVER () AS cdf_treated
            FROM value_counts
        )
        SELECT MAX(ABS(cdf_treated - cdf_control)) FROM cumulative
    ''').fetchone()[0]

balance_rows = []
for f in FEATURES:
    mean_treated = by_arm[1][f + '__mean']
    mean_control = by_arm[0][f + '__mean']
    var_treated = by_arm[1][f + '__var']
    var_control = by_arm[0][f + '__var']

    # Standardized mean difference, written out rather than hidden in a helper.
    pooled_sd = math.sqrt((var_treated + var_control) / 2)
    smd = (mean_treated - mean_control) / pooled_sd if pooled_sd > 0 else float('nan')
    variance_ratio = var_treated / var_control if var_control > 0 else float('nan')

    balance_rows.append({
        'feature': f,
        'mean (treated)': mean_treated,
        'mean (control)': mean_control,
        'SMD': smd,
        'abs SMD': abs(smd),
        'variance ratio': variance_ratio,
        'median gap ~': by_arm[1][f + '__median'] - by_arm[0][f + '__median'],
        'KS distance': ks_distance(f),
    })

balance = pd.DataFrame(balance_rows)
display(balance)

worst_smd = balance.loc[balance['abs SMD'].idxmax()]
worst_ks = balance.loc[balance['KS distance'].idxmax()]

display(Markdown(f'''
**What this shows.** The largest imbalance in means is an absolute SMD of
**{worst_smd['abs SMD']:.4f}** on `{worst_smd['feature']}`; the largest distributional gap is a KS
distance of **{worst_ks['KS distance']:.4f}** on `{worst_ks['feature']}`. Variance ratios lie between
**{balance['variance ratio'].min():.3f}** and **{balance['variance ratio'].max():.3f}**.

**What this does not show.** This cannot prove the assignment was random, cannot show the arms are
comparable on anything unmeasured, and cannot rule out differences confined to regions of the
feature space too small to move a global average. Balance is supporting evidence for a claim about
the study design, never a substitute for one.

**Why it matters next.** Had these numbers been large, the raw difference in conversion rates from
section 4 would be uninterpretable. They are small, which is consistent with treatment having been
randomly assigned — and that is what makes uplift modeling a sensible thing to attempt here.
'''))

In [ ]:
# The same balance evidence, drawn.
smd_ordered = balance.sort_values('SMD')
ks_ordered = balance.sort_values('KS distance')

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))

axes[0].barh(smd_ordered['feature'], smd_ordered['SMD'], color='#4C78A8')
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].axvline(0.10, color='#E45756', linestyle='--', linewidth=1,
                label='plus or minus 0.10 rule-of-thumb reference')
axes[0].axvline(-0.10, color='#E45756', linestyle='--', linewidth=1)
axes[0].set_xlabel('Standardized mean difference (treated minus control)')
axes[0].set_title('Difference in feature means between groups')
axes[0].legend(loc='lower right', fontsize=8)

axes[1].barh(ks_ordered['feature'], ks_ordered['KS distance'], color='#72B7B2')
axes[1].set_xlabel('Largest gap between the two groups cumulative distributions')
axes[1].set_title('Difference in feature shape between groups')

fig.tight_layout()
plt.show()

## 7. How many rows are indistinguishable, and does precision change that?

Some rows carry exactly the same twelve feature values. A model literally cannot tell them apart, so it is worth knowing how common that is.

It is tempting to call these *duplicate users* and delete them. That would be a mistake here, for two independent reasons:

1. **There is no person identifier.** Identical anonymous features are not evidence of the same person. With twelve coarse features spread over roughly 14 million rows, collisions are expected even if every single row is a different individual.
2. **Deleting them changes the question.** The released population *is* the population the conclusions are about. Dropping rows because their values repeat quietly redefines what is being estimated, and does so in a way that is invisible in the final numbers.

So every row is kept. Repeated profiles are measured and reported, never removed. Rows that share `f0`–`f11` but differ in treatment or outcome are especially not contradictions — they are exactly the comparisons an uplift model learns from.

A related question worth separating out: does storing the features at *lower numeric precision* create collisions that are not really in the data? The second cell below groups rows at full 64-bit precision and again after casting to 32-bit.

In [ ]:
# How many rows share an identical profile, under three definitions of identical?
def profile_groups(label, columns, relation='data'):
    grouped = ', '.join(columns)
    row = con.execute(f'''
        WITH groups AS (
            SELECT hash({grouped}) AS profile, COUNT(*) AS n
            FROM {relation}
            GROUP BY profile
        )
        SELECT COUNT(*) FILTER (WHERE n > 1),
               COALESCE(SUM(n) FILTER (WHERE n > 1), 0),
               COALESCE(SUM(n - 1) FILTER (WHERE n > 1), 0),
               MAX(n)
        FROM groups
    ''').fetchone()
    return {
        'grouped by': label,
        'repeated groups': int(row[0]),
        'rows in repeated groups': int(row[1]),
        'rows beyond the first': int(row[2]),
        'largest group': int(row[3]),
        'share of all rows': int(row[1]) / total_n,
    }

duplicates = pd.DataFrame([
    profile_groups('the 12 features only', FEATURES),
    profile_groups('features plus treatment', FEATURES + [TREATMENT]),
    profile_groups('features plus treatment plus conversion', FEATURES + [TREATMENT, OUTCOME]),
])
display(duplicates)

feature_only = duplicates.iloc[0]

display(Markdown(f'''
**What this shows.** Grouping on the twelve features alone,
**{feature_only['rows in repeated groups']:,} rows** ({feature_only['share of all rows']:.2%} of the
data) share their profile with at least one other row, across
**{feature_only['repeated groups']:,}** groups; the largest single group holds
**{feature_only['largest group']:,}** rows. Adding treatment, and then conversion, to the grouping
key splits these into progressively smaller groups.

**What this does not show.** Not one of these numbers counts *people*. Without a user identifier
there is no way to separate one person appearing repeatedly from many people who happen to look
identical on twelve anonymous features. Grouping uses a 64-bit hash, so a negligible number of these
groups could be hash collisions rather than true value matches.

**Why it matters next.** Repeated profiles carrying different outcomes are informative rather than
contradictory — they are direct evidence about how outcomes vary at a fixed feature profile. Every
row is therefore retained.
'''))

In [ ]:
# Does 32-bit storage invent collisions that are not present at full precision?
float32_relation = (
    '(SELECT ' + ', '.join(f'CAST({f} AS FLOAT) AS {f}' for f in FEATURES) + ' FROM data)'
)

precision = pd.DataFrame([
    profile_groups('64-bit precision (used throughout)', FEATURES),
    profile_groups('32-bit precision (comparison only)', FEATURES, relation=float32_relation),
])
display(precision[['grouped by', 'repeated groups', 'rows in repeated groups', 'largest group']])

rows_64 = int(precision.iloc[0]['rows in repeated groups'])
rows_32 = int(precision.iloc[1]['rows in repeated groups'])
extra = rows_32 - rows_64
extra_note = (
    'Any extra collisions here are an artifact of rounding, not a property of the underlying data.'
    if extra > 0 else
    'No extra collisions appeared, but that is a property of these particular values and would not '
    'necessarily hold for a different dataset or a different cast.'
)

display(Markdown(f'''
**What this shows.** At 64-bit precision **{rows_64:,}** rows sit in a repeated profile. Casting the
same features to 32-bit changes that to **{rows_32:,}** — a difference of **{extra:,} rows**
({extra / total_n:.4%} of the data).

**What this does not show.** {extra_note} This comparison says nothing about whether 32-bit storage
would change any model predictions; only whether it changes which rows are distinguishable in the
first place.

**Why it matters next.** Because rounding can merge genuinely distinct rows, the analysis keeps the
features at full 64-bit precision throughout. Lower precision is treated strictly as a sensitivity
comparison, never as the primary representation.
'''))

## 8. What this means for uplift modeling

None of the above is a result about a model — no model has been fit. Each observation is a *constraint* that the modeling work has to respect, and stating these in advance is what stops a later result from being explained after the fact.

**Treatment imbalance.** The arms differ substantially in size. A T-Learner fits one outcome model per arm, so its two halves are trained on very different amounts of data and will not be equally precise. An X-Learner is designed partly for this situation — it borrows strength from the larger arm to estimate effects in the smaller one — which makes it a natural comparison to run, not a guaranteed winner.

**Outcome rarity.** Conversion is rare, so it is the count of *conversions*, not the count of rows, that limits precision. A causal forest partitions the data into leaves; with an outcome this rare, some leaves will contain very few conversions, and effect estimates there will be correspondingly unstable.

**Anonymous features and repeated profiles.** No domain knowledge can be brought to bear on `f0`–`f11`, so any feature engineering has to be driven by observed behaviour rather than by meaning. Identical profiles with differing outcomes are signal, not noise to be cleaned away.

**Global support is not local support.** Both arms exist overall, and each feature range overlaps between them. That is a statement about twelve one-dimensional views. It does *not* establish that both arms are well represented in every region of the twelve-dimensional space, and a model predicting an effect in a sparsely populated corner is extrapolating.

**The fundamental limit.** For every row we observe only one of the two possible outcomes — whatever happened under the assignment that row actually received. The other is permanently unavailable. There is therefore no per-row ground truth against which any prediction can be scored. A predicted uplift is an estimate of an *average* effect among people with similar features; it is never that row true individual effect, and no amount of data on this dataset changes that.

## Key findings

- **Ranking by conversion probability and ranking by treatment effect are different problems.** The customers most likely to convert can be among the least worth treating.
- **The experiment is large and complete.** Every field used here is fully populated, so the analysis runs on the whole released population with nothing dropped.
- **The treatment split is heavily lopsided.** The smaller arm sets the precision ceiling for anything estimated within it.
- **Conversion is rare.** The number of conversions, not the number of rows, is the binding constraint, and it limits how finely the data can be sliced.
- **All four combinations of treatment and conversion occur**, so the basic comparisons an uplift model needs are supported at the global level.
- **The treated and control groups look alike** across means, variances, and distribution shapes. This is consistent with random assignment but does not prove it, and no threshold was used to decide it.
- **A meaningful share of rows are indistinguishable on the twelve features**, and 32-bit rounding shifts that count, which is why full 64-bit precision is used and why no row is removed for repeating.
- **Individual treatment effects are not measurable here**, since only one outcome is ever observed per row. Everything downstream estimates conditional averages, and should be read that way.

## Scope of this notebook

This notebook currently contains the descriptive groundwork above: allocation, outcome rarity, feature shape, balance, and repeated-profile diagnostics. The frozen train/validation/held-out split and its stratification, and the preprocessing/feature-engineering strategy, are separate pieces of work that land in later sections of this same notebook as they are implemented — they are not yet present here, and nothing above should be read as depending on them.